In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/mini-clip-dlai'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Project folder:', DRIVE_DIR)

---
## Clone repo & install dependencies



In [ ]:

!git clone https://github.com/prajwl7676/mini-clip-DLAI.git

%cd /content/mini-clip-DLAI

!pip install -q -r requirements.txt

print('Setup complete.')

---
##Imports & device setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from datasets import load_dataset
from transformers import DistilBertTokenizer
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')
if device.type == 'cuda':
    print(f'GPU             : {torch.cuda.get_device_name(0)}')


##Load Flickr30k from HuggingFace (10k samples)


In [ ]:

##Load Flickr30k from HuggingFace (10k samples)

full_dataset = load_dataset('nlphuji/flickr30k', split='test')

print(f'Total samples in dataset : {len(full_dataset)}')
print(f'Column names             : {full_dataset.column_names}')
print(f'Features                 : {full_dataset.features}')


## Inspect one example

In [ ]:
example = full_dataset[0]

print('Keys in one example:', list(example.keys()))

print('Image type :', type(example['image']))
print('Image size :', example['image'].size)
print('Image mode :', example['image'].mode)
print()
print('Number of captions:', len(example['caption']))
print('Captions:')
for i, cap in enumerate(example['caption']):
    print(f'  [{i}] {cap}')


## Visualise 6 examples


In [ ]:
import os

os.makedirs('assets', exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, ax in enumerate(axes):
    item    = full_dataset[i]
    image   = item['image']
    caption = item['caption'][0]     # caption 0 out of 5

    ax.imshow(image)
    ax.set_title(
        caption if len(caption) <= 60 else caption[:57] + '...',
        fontsize=9, wrap=True
    )
    ax.axis('off')

plt.suptitle('Flickr30k — 6 example image-caption pairs', fontsize=12)
plt.tight_layout()
plt.savefig('assets/flickr_examples.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to assets/flickr_examples.png')

---
## Split into train / val / test-- 80 / 10 / 10.


In [ ]:
TOTAL  = 10_000
N_TRAIN = 8_000
N_VAL   = 1_000
# test = remaining 1,000

dataset_10k = full_dataset.select(range(TOTAL))

train_hf = dataset_10k.select(range(0, N_TRAIN))
val_hf   = dataset_10k.select(range(N_TRAIN, N_TRAIN + N_VAL))
test_hf  = dataset_10k.select(range(N_TRAIN + N_VAL, TOTAL))

print(f'Train : {len(train_hf):,} samples')
print(f'Val   : {len(val_hf):,} samples')
print(f'Test  : {len(test_hf):,} samples')


##  Load the tokenizer

The tokenizer converts a caption string into a sequence of integer IDs  
that the DistilBERT text encoder can process.


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# --- see what tokenisation looks like ---
sample_caption = train_hf[0]['caption'][0]
sample_image= train_hf[0]['image']
plt.imshow(sample_image)
plt.axis('off')
plt.show()

print('Image size :', sample_image.size)
print('Caption :', sample_caption)
print()



In [ ]:
encoded = tokenizer(
    sample_caption,
    padding='max_length',
    max_length=77,
    truncation=True,
    return_tensors='pt',
)
print('input_ids shape      :', encoded['input_ids'].shape)       # (1, 77)
print('attention_mask shape :', encoded['attention_mask'].shape)  # (1, 77)
print()
print('input_ids (first 15 tokens):', encoded['input_ids'][0, :15].tolist())
print('Decoded back         :', tokenizer.decode(encoded['input_ids'][0]))

---
##  Define image transforms

ResNet-50 expects **224 × 224** images, normalised with ImageNet statistics.  
Using slightly different transforms for training (with augmentation) vs eval.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# Test on one image
sample_image = train_hf[0]['image'].convert('RGB')
tensor = train_transform(sample_image)

print('Original image size :', sample_image.size)   # (W, H)
print('After transform     :', tensor.shape)         # (3, 224, 224)
print('Pixel value range   : [{:.2f}, {:.2f}]'.format(tensor.min().item(), tensor.max().item()))

---
## Write the FlickrDataset class

 `src/dataset.py`.  


In [ ]:
class FlickrDataset(Dataset):


    def __init__(self, hf_dataset, tokenizer, transform, max_length=77, caption_idx=0):
        self.data        = hf_dataset
        self.tokenizer   = tokenizer
        self.transform   = transform
        self.max_length  = max_length
        self.caption_idx = caption_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Image
        image = item['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        image = self.transform(image)          # (3, 224, 224)

        # Caption
        captions = item['caption']
        caption  = captions[self.caption_idx] if isinstance(captions, list) else captions

        # Tokenise
        encoded = self.tokenizer(
            caption,
            padding='max_length',
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt',
        )
        tokens = {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
        }
        return image, tokens


print('FlickrDataset class defined.')

---
## Test the Dataset

Before building a DataLoader, verify that a single `__getitem__` call  
returns the correct shapes.

In [ ]:
train_dataset = FlickrDataset(train_hf, tokenizer, train_transform)
val_dataset   = FlickrDataset(val_hf,   tokenizer, eval_transform)
test_dataset  = FlickrDataset(test_hf,  tokenizer, eval_transform)

print(f'Train dataset size : {len(train_dataset):,}')
print(f'Val   dataset size : {len(val_dataset):,}')
print(f'Test  dataset size : {len(test_dataset):,}')


In [ ]:
# Fetch one item and check shapes
image, tokens = train_dataset[0]

print('image shape        :', image.shape)                    # (3, 224, 224)
print('input_ids shape    :', tokens['input_ids'].shape)      # (77,)
print('attention_mask shape:', tokens['attention_mask'].shape) # (77,)
print('image dtype        :', image.dtype)                    # torch.float32
print('input_ids dtype    :', tokens['input_ids'].dtype)      # torch.int64

# Sanity check
assert image.shape       == (3, 224, 224), 'Wrong image shape!'
assert tokens['input_ids'].shape == (77,), 'Wrong token shape!'
print()
print('All shape checks passed.')

---
##  Build DataLoaders and test a batch

A `DataLoader` wraps a `Dataset` and handles batching, shuffling,  
and parallel loading. `batch_size=64`

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,    # faster CPU→GPU memory transfer
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

In [ ]:
# Fetch one batch and verify shapes
images_batch, tokens_batch = next(iter(train_loader))

print('--- One batch from train_loader ---')
print('images shape       :', images_batch.shape)                     # (64, 3, 224, 224)
print('input_ids shape    :', tokens_batch['input_ids'].shape)         # (64, 77)
print('attention_mask shape:', tokens_batch['attention_mask'].shape)   # (64, 77)


---
##  Visualise a batch (un-normalise for display)

After normalisation, pixel values are no longer in [0, 1] —
reverse the transform to display images correctly.

In [ ]:
def unnormalise(tensor):
    """Reverse ImageNet normalisation for display. Input: (3, H, W) tensor."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img_tensor = unnormalise(images_batch[i])        # (3, 224, 224)
    img_np     = img_tensor.permute(1, 2, 0).numpy() # (224, 224, 3)

    # Decode the caption tokens back to text
    ids     = tokens_batch['input_ids'][i]
    caption = tokenizer.decode(ids, skip_special_tokens=True)
    caption = caption[:55] + '...' if len(caption) > 55 else caption

    ax.imshow(img_np)
    ax.set_title(caption, fontsize=8)
    ax.axis('off')

plt.suptitle('Batch sample — images with decoded captions', fontsize=11)
plt.tight_layout()
plt.savefig('assets/batch_sample.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to assets/batch_sample.png')

---
##  Caption length analysis

Understanding caption lengths helps choose `max_length=77` wisely.  


In [ ]:
# Sample 500 captions and measure their token lengths
lengths = []
for i in range(500):
    cap = train_hf[i]['caption'][0]
    toks = tokenizer(cap, truncation=False)['input_ids']
    lengths.append(len(toks))

lengths = np.array(lengths)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(77, color='crimson', linestyle='--', linewidth=1.5, label='max_length=77')
ax.set_xlabel('Caption length (tokens)')
ax.set_ylabel('Count')
ax.set_title('Distribution of caption token lengths (500 samples)')
ax.legend()
plt.tight_layout()
plt.savefig('assets/caption_lengths.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Mean   : {lengths.mean():.1f} tokens')
print(f'Median : {np.median(lengths):.1f} tokens')
print(f'Max    : {lengths.max()} tokens')
print(f'% > 77 : {(lengths > 77).mean() * 100:.1f}%')

---
##  Extract to src/dataset.py



In [ ]:
import sys
sys.path.insert(0, '/content/mini-clip-DLAI')

from src.dataset import FlickrDataset, build_loaders, get_transform

In [ ]:
train_ds = FlickrDataset(train_hf, tokenizer, get_transform(train=True))
val_ds   = FlickrDataset(val_hf,   tokenizer, get_transform(train=False))
test_ds  = FlickrDataset(test_hf,  tokenizer, get_transform(train=False))

train_loader, val_loader, test_loader = build_loaders(
    train_ds, val_ds, test_ds, batch_size=64
)

imgs, toks = next(iter(train_loader))
assert imgs.shape == (64, 3, 224, 224)
assert toks['input_ids'].shape == (64, 77)

print('src.dataset import works correctly.')
